# End-to-End Evaluation: RAGAS Metrics and 128 K Long-Context Baseline

**Yusif Askari** &middot; SCIPER 413862 &middot; CS-552 Spring 2026 &middot; Team CiteRight

Scores every pipeline variant with RAGAS (faithfulness, answer relevancy, context precision/recall) and benchmarks them against a long-context baseline that stuffs full papers into a 128 K window. Reports the cost / latency / quality trade-off.

---

This notebook is part of the **Faithful RAG** Open Project. It runs end-to-end inside the RCP pod launched by [`notebooks/submit.sh`](./submit.sh) with no manual setup: dependencies install in the bootstrap cell, the corpus downloads from [`citeright/corpus`](https://huggingface.co/datasets/citeright/corpus) into `/scratch`, and all generation runs locally on the 40 GB A100 by default. API-based comparisons are optional and skipped if no key is provided.

## Setup

Installs Python dependencies (idempotent on rerun), wires `sys.path` so we can `from evaluation.common import ...`, and resolves the corpus cache. On RCP this finds `/scratch/citeright_artifacts`; on a laptop it falls back to `data/s3_archive/` if you've cloned a local mirror, otherwise downloads from HuggingFace.

In [ ]:
%pip install --quiet huggingface_hub sentence-transformers faiss-cpu datasets transformers accelerate litellm tqdm pandas matplotlib

In [ ]:
import logging
import os
import sys
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# Locate repo root (notebook lives in <repo>/notebooks/).
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name == 'notebooks' or not (REPO_ROOT / 'pyproject.toml').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('Could not find repo root (no pyproject.toml above notebook).')
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# On RCP submit.sh sets these; default for laptop dev otherwise.
os.environ.setdefault('CITERIGHT_HF_REPO', 'citeright/corpus')
if Path('/scratch').is_dir():
    os.environ.setdefault('CITERIGHT_DATA_DIR', '/scratch/citeright_artifacts')
    os.environ.setdefault('HF_HOME', '/scratch/hf_cache')

from evaluation.common import (
    artifact_root, available_models, generate, 
    load_chunk_metadata, load_faiss_index, load_gold_qa, load_paper_markdown,
)

ART = artifact_root()
print(f'Corpus root: {ART}')
print(f'Available models: {len(available_models())}')

### Optional: API keys

The notebook works with **only** local models. If you want the OpenAI / OpenRouter comparison arms (e.g. GPT-5, Claude, DeepSeek), paste the keys when prompted below. Leave blank to skip.

In [ ]:
# Optional: paid-API comparison arms.
# Skip these cells if you only want the local-model path (always works).
import getpass
for var in ('OPENAI_API_KEY', 'OPENROUTER_API_KEY'):
    if var not in os.environ:
        val = getpass.getpass(f'{var} (leave blank to skip): ').strip()
        if val:
            os.environ[var] = val
print('Local models always available; API arms enabled iff keys above were provided.')

## Smoke test

Loads the FAISS index, the row-→-chunk metadata, and the gold evaluation set. If this cell errors, the rest of the notebook will not run — see `evaluation/common/data_loader.py` for the resolution rules.

In [ ]:
# Confirm the corpus loads end-to-end.
import faiss

meta = load_chunk_metadata('coarse')
index = load_faiss_index('coarse')
gold = load_gold_qa()

print(f'Coarse FAISS rows: {index.ntotal}')
print(f'Coarse metadata entries: {len(meta)}')
print(f'Gold Q&A pairs: {len(gold)}')
assert index.ntotal == len(meta), 'FAISS / metadata size mismatch'

# Show one chunk so we know the schema.
first = meta[0]
print(f"\nExample chunk:\n  paper_id: {first['paper_id']}")
print(f"  section : {first.get('section_hierarchy', [])}")
print(f"  text    : {first['text'][:200]}...")

## Run the full pipeline on the gold set

For each generator, collect `(question, answer, contexts, ground_truth)` tuples. Use the same prompt template across configurations so RAGAS scores are comparable.

In [ ]:
# TODO: implement
raise NotImplementedError

## RAGAS scoring

Build a `datasets.Dataset` from the tuples and call `ragas.evaluate(...)` with the four metrics. Cache scores to `evaluation/ragas_eval/results/`.

In [ ]:
# TODO: implement
raise NotImplementedError

## Long-context baseline

For each gold question identify the relevant papers, concatenate their `document.md` (truncating to 128 K tokens), and answer in one shot with a 128 K-context model. Score with the same RAGAS metrics.

In [ ]:
# TODO: implement
raise NotImplementedError

## Cost / latency / accuracy trade-off

Plot RAGAS faithfulness vs (i) tokens per query and (ii) wall-clock latency. Compare chunked-RAG configurations to the long-context baseline. Discuss when each wins.

In [ ]:
# TODO: implement
raise NotImplementedError

## Results & discussion

Summarise the headline numbers, ablations, and comparisons here. The 4-page report references plots and tables produced above.

## Reproduction

From a clean clone of the repo:

```bash
cd notebooks && ./submit.sh         # launch the RCP pod
runai port-forward <job> --port 8888:8888
# open http://localhost:8888 (token: cs552), then Run All on this notebook
```

All artefacts download automatically from [`citeright/corpus`](https://huggingface.co/datasets/citeright/corpus); no S3 or AWS access is required.